# Lektion 14: Hybrid Retrieval – Vektor trifft Graph



| | |
|---|---|
| **Zeit** | ~60 Minuten |
| **Schwierigkeit** | Mittel |
| **Voraussetzungen** | Lektion 08–13 |
| **Du baust** | Die Messung: dieselben Fragen an beide Wissenssysteme – und die Hybrid-Pipeline mit Quellentrennung |

## 🎯 Nach dieser Lektion kannst du:

- Vektor- und Graph-Retrieval **unabhängig** betreiben

- die Stärken beider Verfahren an konkreten Fragen messen

- den **Emergenz-Beweis** wiederholen (Pfad, der nirgends geschrieben steht)

- eine Hybrid-Pipeline bauen, die Doku-Fundstellen und Graph-Fakten **getrennt** ausweist

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [ ]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


## 🧠 Konzept: Zwei Abteilungen der Bibliothek

**Metapher:**
- **Vektor-Suche** = der hilfsbereite **Ausleiher**: "Ich glaube, *dieses* Kapitel meinst du" – unscharf, nach Bedeutung. Perfekt für "Was steht über…?"
- **Graph-Suche** = der **Archivar** mit Fadensystem: "Alles zu X hängt an *diesem* Faden" – exakt, nach Struktur. Perfekt für "Wie hängt A mit B zusammen?"
- **Hybrid** = frag beide, lege beide Ergebnisse mit **Herkunftsetikett** auf den Tisch.

| Frage | Gewinner |
|---|---|
| "Was steht über Urlaub?" | Vektor |
| "Wer hängt am Projekt?" | Graph |
| "Wie hängt Ben mit dem Projekt zusammen?" | Graph (Emergenz!) |
| "Was wissen wir gesamt?" | Hybrid |

In [ ]:
# Beide Systeme aufbauen (Wiederverwendung aus Lektion 08/10!):
from lernakademie.build2 import baue_retriever
from lernakademie.graph_store import GraphStore

retriever = baue_retriever(k=3)

def baue_wissens_graph():
    store = GraphStore()
    anna = store.add_node(["Person"], name="Anna", rolle="Entwicklerin")
    ben = store.add_node(["Person"], name="Ben", rolle="Analyst")
    engineering = store.add_node(["Department"], name="Engineering")
    analytics = store.add_node(["Department"], name="Data & Analytics")
    projekt = store.add_node(["Project"], name="AI Assistant", status="aktiv")
    store.add_relationship(anna, engineering, "BELONGS_TO")
    store.add_relationship(ben, analytics, "BELONGS_TO")
    store.add_relationship(engineering, projekt, "WORKS_ON")
    store.add_relationship(analytics, projekt, "PROVIDES_DATA_FOR")
    store.add_relationship(anna, projekt, "CONTRIBUTES_TO")
    return store

graph = baue_wissens_graph()
print("Beide Systeme bereit:",
      type(retriever).__name__, "+", len(graph.nodes()), "Graph-Nodes")

In [ ]:
def vektor_suche(frage):
    """Liefert [{quelle, textstelle}] – nur relevante Treffer."""
    from lernakademie.offline_embeddings import hat_bezug
    from pathlib import Path
    if not hat_bezug(frage):
        return []
    ergebnisse = []
    for dokument in retriever.invoke(frage)[:2]:
        stelle = dokument.page_content.strip().replace("\n", " ")
        if len(stelle) > 30:
            ergebnisse.append({"quelle": Path(dokument.metadata["source"]).name,
                               "textstelle": stelle[:140] + "..."})
    return ergebnisse

def graph_suche(frage):
    """Liefert Fakten der Form 'A --REL--> B' (Traversierung)."""
    klein = frage.lower()
    fakten = []
    projekt = graph.find_node("Project", name="AI Assistant")
    if projekt and ("projekt" in klein or "ai assistant" in klein):
        for kante in graph.relationships():
            if kante.to_id == projekt.id:
                quelle = graph.nodes()[kante.from_id]
                fakten.append(f"{quelle.properties['name']} --{kante.typ}--> AI Assistant")
    if "ben" in klein:
        ben = graph.find_node("Person", name="Ben")
        for kante, abteilung in graph.neighbors(ben.id, "BELONGS_TO"):
            for kante2, ziel in graph.neighbors(abteilung.id):
                fakten.append(f"Ben --BELONGS_TO--> {abteilung.properties['name']}"
                              f" --{kante2.typ}--> {ziel.properties['name']}")
    return list(dict.fromkeys(fakten))   # Duplikate raus, Reihenfolge halten

print("Vektor:", [e["quelle"] for e in vektor_suche("Wie viele Urlaubstage gibt es?")])
print("Graph :", graph_suche("Wer haengt am AI Assistant dran?"))

## 🔎 Der direkte Vergleich

Jetzt **dieselben Fragen** an **beide** Systeme – und die Auswertung:

In [ ]:
VERGLEICH = [
    ("Was steht ueber die Abteilung Engineering?", "Vektor (Textinhalt)"),
    ("Wer haengt am AI Assistant dran?", "Graph (Verbindungen)"),
    ("Wie haengt Ben mit dem Projekt zusammen?", "Graph (Emergenz!)"),
    ("Was wissen wir gesamt ueber das Projekt?", "Beide (Hybrid)"),
]
for frage, erwartet in VERGLEICH:
    v = vektor_suche(frage)
    g = graph_suche(frage)
    print(f"FRAGE: {frage}")
    print(f"  Vektor: {len(v)} Fundstelle(n) | Graph: {len(g)} Fakt(en) | erwartet: {erwartet}")
    for fakt in g[:2]:
        print("   Graph-Fakt:", fakt)
    print()

### 🔎 Der Emergenz-Beweis
`Ben --BELONGS_TO--> Data & Analytics --PROVIDES_DATA_FOR--> AI Assistant`: Dieser Weg steht in **keinem Satz** der Dokumente – er entsteht durch das Kombinieren zweier Fakten. Vektor-Suche könnte ihn maximal *andeutungsweise* streifen; der Graph beantwortet ihn **exakt**.

## ✏️ Deine Aufgabe: Die Hybrid-Pipeline

Du baust `hybrid_suche(frage)`, das beide Quellen befragt – sauber getrennt und mit ehrlichem Fehlschlag.

**Anforderungen:**

- Rückgabe: `{"dokumente": [...], "graph": [...], "antwort": str}`.

- Beide Quellen werden immer befragt (das Routing folgt in Lektion 15!).

- Wenn beide leer sind: `antwort = "Dazu habe ich in Dokumenten UND Graph nichts gefunden."`

- Die Antwort nennt die Anzahl der Fundstellen je Quelle.

**Schreibe deinen Code in die folgende Zelle:**

In [ ]:
def hybrid_suche(frage):
    # YOUR CODE HERE
    pass

ergebnis = hybrid_suche("Was wissen wir gesamt ueber das Projekt?")
print(ergebnis["antwort"])

### ✅ Richtig, wenn:

- [ ] Für die Gesamtfrage liefert die Pipeline Daten aus **beiden** Feldern.

- [ ] Totaler Fehlschlag → ehrlicher Text, keine Crashes.

- [ ] Doku- und Graph-Quellen sind in getrennten Feldern (nie vermischt!).

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [ ]:
hy = hybrid_suche("Was wissen wir gesamt ueber das AI Assistant Projekt?")
assert hy["dokumente"] or hy["graph"], "Hybrid-Frage sollte Treffer haben."
assert isinstance(hy["dokumente"], list) and isinstance(hy["graph"], list)
leer = hybrid_suche("Gibt es hier Informationen zu Weihnachtsgeld?")
assert leer["antwort"], "Auch leer braucht eine Antwort."
print("Alle Prüfungen bestanden! ✅")

<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Baue auf `vektor_suche(frage)` und `graph_suche(frage)` auf – die Arbeit ist schon getan!
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Der Antwort-Text kann deterministisch zusammengesetzt werden (kein LLM nötig): `f"Hybrid: {len(d)} Doku- und {len(g)} Graph-Treffer."`
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Warum wäre "immer beide Quellen fragen" bei 100 Nutzern zu teuer – und was rettet uns? (→ Routing!)

2. Was fehlt der Hybrid-Antwort, wenn der Graph **falsch gepflegt** wurde – und was schützt dagegen? (→ Freigabe, Lektion 15!)

## ➡️ Weiter

**🚀 BUILD 3 – Lektion 15:** `notebooks/08_build_3/15_build3_hybrid_agent.ipynb` – LangGraph verbindet alles: Routing zwischen den Quellen UND die Freigabe für Graph-Änderungen.